In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [ ]:
"""
EVA02-Enormous — 4-Seed Ensemble Inference
============================================
Seeds: 42, 62, 82, 120

Soft-vote (arithmetic mean) over 4 independently trained models.
Each model gets 8-way geometric TTA → 32 total forward passes.

Also tries geometric mean (emphasises model agreement) and picks
whichever scores higher on the shared 516-sample val set.

Outputs:
  - T2_enormous_ensemble_submission.csv
  - enormous_ensemble_probs.npy (for further ensembling with Large models)
  - Val F1 comparison: single-model vs ensemble
"""

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

try:
    import timm, albumentations
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                    'timm', 'albumentations'])

import gc, time, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix

warnings.filterwarnings('ignore')


# ╔══════════════════════════════════════════════════════════════╗
# ║                     PATHS — FILL THESE IN                   ║
# ╚══════════════════════════════════════════════════════════════╝
class CFG:
    # -------- DATA --------
    data_root = '/kaggle/input/PUT-YOUR-CROP-DATASET'           # <-- UPDATE
    crops_dir = os.path.join(data_root, 'crops')
    master_csv = os.path.join(data_root, 'master.csv')

    # -------- PRETRAINED BACKBONE (same for all seeds) --------
    pretrained_weights = '/kaggle/input/PUT-YOUR-PRETRAINED'    # <-- UPDATE

    # -------- SEED CHECKPOINTS (best.pth from each seed's training) --------
    seed_checkpoints = {
        42:  '/kaggle/input/PUT-SEED-42/eva02e_lpft_best.pth',  # <-- UPDATE
        62:  '/kaggle/input/PUT-SEED-62/eva02e_lpft_best.pth',  # <-- UPDATE
        82:  '/kaggle/input/PUT-SEED-82/eva02e_lpft_best.pth',  # <-- UPDATE
        120: '/kaggle/input/PUT-SEED-120/eva02e_lpft_best.pth', # <-- UPDATE
    }

    # -------- MODEL ARCHITECTURE (must match training exactly) --------
    model_name = 'eva02_enormous_patch14_clip_224.laion2b_plus'
    img_size    = 336
    num_classes = 5
    total_blocks = 64
    unfreeze_last_n = 24
    drop_path_rate = 0.35          # match training
    head_hidden_dim = 512
    head_dropout    = 0.30
    head_mid_dropout = 0.20

    # -------- INFERENCE --------
    batch_size  = 32               # safe on 102 GB; lower if OOM
    num_workers = 10
    tta_modes   = 8                # 8 = full geometric TTA
    gradient_checkpointing = False # not needed for inference


# ╔══════════════════════════════════════════════════════════════╗
# ║                        CONSTANTS                            ║
# ╚══════════════════════════════════════════════════════════════╝
TEST_CAMERAS = ['pen1_tur_cam1', 'pen2_orb_cam2', 'pen2_tur_cam2']
CN = {0: 'LatL', 1: 'LatR', 2: 'Sit', 3: 'Stand', 4: 'Stern'}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16

if torch.cuda.is_available():
    n = torch.cuda.get_device_name(0)
    v = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'[DEVICE] {n} ({v:.0f} GB) | bf16: {BF16}', flush=True)


# ╔══════════════════════════════════════════════════════════════╗
# ║                    MODEL + WEIGHT LOADING                   ║
# ╚══════════════════════════════════════════════════════════════╝
def safe_torch_load(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def interpolate_pos_embed(state, backbone):
    if 'pos_embed' not in state or not hasattr(backbone, 'pos_embed'):
        return state
    old_pe = state['pos_embed']
    new_pe = backbone.pos_embed
    if old_pe.shape == new_pe.shape:
        return state
    dim = old_pe.shape[-1]
    cls_tok = old_pe[:, :1, :]
    patch_pe = old_pe[:, 1:, :]
    old_grid = int(patch_pe.shape[1] ** 0.5)
    new_grid = int((new_pe.shape[1] - 1) ** 0.5)
    patch_pe = patch_pe.reshape(1, old_grid, old_grid, dim).permute(0, 3, 1, 2).float()
    patch_pe = F.interpolate(patch_pe, size=(new_grid, new_grid),
                             mode='bicubic', align_corners=False)
    patch_pe = patch_pe.permute(0, 2, 3, 1).reshape(1, new_grid * new_grid, dim)
    state['pos_embed'] = torch.cat([cls_tok, patch_pe], dim=1)
    return state


class PigModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.model_name, pretrained=False, num_classes=0,
            img_size=CFG.img_size, drop_path_rate=CFG.drop_path_rate,
        )
        # Load pretrained backbone
        if CFG.pretrained_weights and os.path.exists(CFG.pretrained_weights):
            st = safe_torch_load(CFG.pretrained_weights)
            # Strip common prefixes
            for prefix in ('module.', 'backbone.', 'model.'):
                st = {(k[len(prefix):] if k.startswith(prefix) else k): v
                      for k, v in st.items()}
            st = {k: v for k, v in st.items()
                  if not k.startswith('head.') and not k.startswith('fc_norm.')}
            st = interpolate_pos_embed(st, self.backbone)
            self.backbone.load_state_dict(st, strict=False)
            del st; gc.collect()

        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(CFG.head_dropout),
            nn.Linear(feat_dim, CFG.head_hidden_dim),
            nn.GELU(),
            nn.Dropout(CFG.head_mid_dropout),
            nn.Linear(CFG.head_hidden_dim, CFG.num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def load_seed_weights(model, path):
    """Load best.pth trainable_state into model (same as inference script)."""
    ckpt = safe_torch_load(path)
    state = ckpt.get('trainable_state', ckpt)
    best_f1 = ckpt.get('best_f1', 0.0)
    best_ep = ckpt.get('best_ep', 0)
    use_ema = ckpt.get('use_ema', False)

    # Set stage 2 trainability so names match
    for p in model.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True
    nb = len(model.backbone.blocks)
    start = max(0, nb - CFG.unfreeze_last_n)
    for i in range(start, nb):
        for p in model.backbone.blocks[i].parameters():
            p.requires_grad = True
    for attr in ('norm', 'fc_norm'):
        if hasattr(model.backbone, attr):
            for p in getattr(model.backbone, attr).parameters():
                p.requires_grad = True

    loaded = 0
    for name, param in model.named_parameters():
        if param.requires_grad and name in state:
            param.data.copy_(state[name].to(device=param.device, dtype=param.dtype))
            loaded += 1

    # Freeze everything for inference
    for p in model.parameters():
        p.requires_grad = False

    del ckpt, state; gc.collect()
    return best_f1, best_ep, use_ema, loaded


# ╔══════════════════════════════════════════════════════════════╗
# ║                         DATASET                             ║
# ╚══════════════════════════════════════════════════════════════╝
def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.img_size, CFG.img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


class PigDS(Dataset):
    def __init__(self, df, crops_dir, tfms):
        self.df = df.reset_index(drop=True)
        self.crops_dir = crops_dir
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(os.path.join(self.crops_dir, row['crop_filename']),
                         cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.tfms:
            img = self.tfms(image=img)['image']
        label = int(row['class_id']) if 'class_id' in row else -1
        return img, label, row['row_id']


# ╔══════════════════════════════════════════════════════════════╗
# ║                      8-WAY TTA ENGINE                       ║
# ╚══════════════════════════════════════════════════════════════╝
def apply_tta(imgs, mode):
    """Returns (transformed_images, needs_lr_swap)."""
    if mode == 0: return imgs, False                                        # original
    if mode == 1: return torch.flip(imgs, [3]), True                        # hflip
    if mode == 2: return torch.flip(imgs, [2]), False                       # vflip
    if mode == 3: return torch.flip(imgs, [2, 3]), True                     # hflip+vflip
    if mode == 4: return torch.rot90(imgs, 1, [2, 3]), False                # 90°
    if mode == 5: return torch.rot90(imgs, 2, [2, 3]), False                # 180°
    if mode == 6: return torch.rot90(imgs, 3, [2, 3]), False                # 270°
    if mode == 7: return torch.flip(torch.rot90(imgs, 1, [2, 3]), [3]), True # 90°+hflip
    return imgs, False


@torch.no_grad()
def run_model_inference(model, loader, n_tta):
    """Run inference with n_tta TTA modes. Returns (row_ids, labels, avg_probs)."""
    model.eval()
    all_ids = []
    all_labels = []
    all_probs = []

    for tta_mode in range(n_tta):
        probs_this = []
        for imgs, labels, rids in loader:
            if tta_mode == 0:
                all_ids.extend(rids)
                all_labels.extend(labels.numpy().tolist())
            imgs_t, needs_swap = apply_tta(imgs.to(DEVICE), tta_mode)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(imgs_t)
            p = torch.softmax(logits.float(), dim=1).cpu().numpy()
            if needs_swap:
                p[:, [0, 1]] = p[:, [1, 0]]
            probs_this.append(p)
        all_probs.append(np.concatenate(probs_this, axis=0))

    avg_probs = np.mean(all_probs, axis=0)
    return all_ids, np.array(all_labels), avg_probs


# ╔══════════════════════════════════════════════════════════════╗
# ║                    ENSEMBLE STRATEGIES                      ║
# ╚══════════════════════════════════════════════════════════════╝
def arithmetic_mean(probs_list):
    """Standard soft voting: average probabilities."""
    return np.mean(probs_list, axis=0)


def geometric_mean(probs_list):
    """Geometric mean: emphasises agreement between models.
    If 3/4 models say class X at 0.9 but 1 model says 0.01,
    geometric mean penalises that disagreement more than arithmetic.
    Re-normalises so rows sum to 1."""
    stacked = np.stack(probs_list, axis=0)
    # Clip to avoid log(0)
    stacked = np.clip(stacked, 1e-7, 1.0)
    log_mean = np.mean(np.log(stacked), axis=0)
    geo = np.exp(log_mean)
    # Re-normalise
    geo = geo / geo.sum(axis=1, keepdims=True)
    return geo


def rank_average(probs_list):
    """Rank averaging: convert probs to ranks per-class, average ranks,
    then convert back. Robust to calibration differences between models."""
    from scipy.stats import rankdata
    n_samples = probs_list[0].shape[0]
    n_classes = probs_list[0].shape[1]
    rank_sum = np.zeros((n_samples, n_classes))
    for probs in probs_list:
        for c in range(n_classes):
            rank_sum[:, c] += rankdata(probs[:, c])
    # Normalise to probabilities
    rank_avg = rank_sum / rank_sum.sum(axis=1, keepdims=True)
    return rank_avg


def evaluate_ensemble(probs, labels, name):
    """Evaluate an ensemble on the val set."""
    preds = probs.argmax(1)
    mask = labels >= 0
    if mask.sum() == 0:
        return 0.0
    f1 = f1_score(labels[mask], preds[mask], average='macro')
    pc = f1_score(labels[mask], preds[mask], average=None, labels=list(range(5)))
    print(f'  {name:>20s}: F1={f1:.4f} | '
          + ' | '.join(f'{CN[i]}:{pc[i]:.3f}' for i in range(5)),
          flush=True)
    return f1


# ╔══════════════════════════════════════════════════════════════╗
# ║                           MAIN                              ║
# ╚══════════════════════════════════════════════════════════════╝
if __name__ == '__main__':
    T0 = time.time()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    seeds = sorted(CFG.seed_checkpoints.keys())
    print('=' * 70, flush=True)
    print(f'  EVA02-Enormous {len(seeds)}-Seed Ensemble', flush=True)
    print(f'  Seeds: {seeds}', flush=True)
    print(f'  TTA: {CFG.tta_modes}-way | batch_size: {CFG.batch_size}', flush=True)
    print('=' * 70, flush=True)

    # ---- 1. Load data ----
    df = pd.read_csv(CFG.master_csv)
    df_tp = df[df['set'].isin(['train1', 'train2'])].copy().reset_index(drop=True)
    df_tp['cg'] = (df_tp['pen'].astype(str) + '_'
                   + df_tp['cam_type'].astype(str) + '_'
                   + df_tp['cam_num'].astype(str))
    df_va = df_tp[df_tp['cg'].isin(TEST_CAMERAS)].reset_index(drop=True)
    df_test = df[df['set'] == 'test'].copy().reset_index(drop=True)

    print(f'\n  Val: {len(df_va)} | Test: {len(df_test)}', flush=True)

    tfms = get_val_transforms()
    va_ds = PigDS(df_va, CFG.crops_dir, tfms)
    va_ld = DataLoader(va_ds, CFG.batch_size, shuffle=False,
                       num_workers=CFG.num_workers, pin_memory=True)

    has_test = len(df_test) > 0
    te_ld = None
    if has_test:
        te_ds = PigDS(df_test, CFG.crops_dir, tfms)
        te_ld = DataLoader(te_ds, CFG.batch_size, shuffle=False,
                           num_workers=CFG.num_workers, pin_memory=True)

    # ---- 2. Run each seed ----
    val_probs_per_seed = {}
    test_probs_per_seed = {}

    for seed in seeds:
        ckpt_path = CFG.seed_checkpoints[seed]
        if not os.path.exists(ckpt_path):
            print(f'\n  ✗ Seed {seed}: checkpoint not found at {ckpt_path}', flush=True)
            continue

        print(f'\n{"─"*60}', flush=True)
        print(f'  Seed {seed}', flush=True)
        print(f'{"─"*60}', flush=True)

        t_seed = time.time()

        # Build model fresh each time (backbone from pretrained, then overlay seed weights)
        model = PigModel().to(DEVICE)
        best_f1, best_ep, use_ema, n_loaded = load_seed_weights(model, ckpt_path)
        tag = 'EMA' if use_ema else 'raw'
        print(f'  Loaded {n_loaded} params | best_f1={best_f1:.4f} (ep{best_ep}, {tag})',
              flush=True)

        # Val inference
        va_ids, va_labels, va_probs = run_model_inference(model, va_ld, CFG.tta_modes)
        val_probs_per_seed[seed] = va_probs

        va_preds = va_probs.argmax(1)
        mask = va_labels >= 0
        seed_f1 = f1_score(va_labels[mask], va_preds[mask], average='macro')
        seed_pc = f1_score(va_labels[mask], va_preds[mask], average=None,
                           labels=list(range(5)))
        print(f'  Val F1 ({CFG.tta_modes}-TTA): {seed_f1:.4f} | '
              + ' | '.join(f'{CN[i]}:{seed_pc[i]:.3f}' for i in range(5)),
              flush=True)

        # Test inference
        if has_test:
            te_ids, _, te_probs = run_model_inference(model, te_ld, CFG.tta_modes)
            test_probs_per_seed[seed] = te_probs

        # Free GPU
        del model
        gc.collect()
        torch.cuda.empty_cache()
        print(f'  Done in {(time.time() - t_seed)/60:.1f} min', flush=True)

    # ---- 3. Ensemble ----
    loaded_seeds = sorted(val_probs_per_seed.keys())
    n_loaded = len(loaded_seeds)

    if n_loaded == 0:
        print('\n[ERROR] No models loaded! Check paths.', flush=True)
        exit(1)

    print(f'\n{"="*70}', flush=True)
    print(f'  ENSEMBLING {n_loaded} SEEDS: {loaded_seeds}', flush=True)
    print(f'{"="*70}', flush=True)

    val_probs_list = [val_probs_per_seed[s] for s in loaded_seeds]
    va_labels = np.array([int(df_va.iloc[i]['class_id']) for i in range(len(df_va))])

    # Try all ensemble strategies on val
    print(f'\n  Val F1 comparison:', flush=True)

    # Individual models
    for s in loaded_seeds:
        evaluate_ensemble(val_probs_per_seed[s], va_labels, f'Seed {s} alone')

    # Ensemble strategies
    arith_probs = arithmetic_mean(val_probs_list)
    arith_f1 = evaluate_ensemble(arith_probs, va_labels, 'Arithmetic Mean')

    geo_probs = geometric_mean(val_probs_list)
    geo_f1 = evaluate_ensemble(geo_probs, va_labels, 'Geometric Mean')

    rank_probs = rank_average(val_probs_list)
    rank_f1 = evaluate_ensemble(rank_probs, va_labels, 'Rank Average')

    # Pick best strategy
    strategies = {
        'arithmetic': (arith_f1, arith_probs),
        'geometric':  (geo_f1,   geo_probs),
        'rank':       (rank_f1,  rank_probs),
    }
    best_name, (best_f1, best_val_probs) = max(strategies.items(),
                                                  key=lambda x: x[1][0])
    print(f'\n  ★ Best strategy: {best_name} (Val F1 = {best_f1:.4f})', flush=True)

    # Val confusion matrix for best strategy
    best_preds = best_val_probs.argmax(1)
    cm = confusion_matrix(va_labels, best_preds, labels=list(range(5)))
    print(f'\n  Confusion Matrix ({best_name}):', flush=True)
    print(f'  {"":>6} ' + ' '.join(f'{CN[i]:>6}' for i in range(5)), flush=True)
    for i in range(5):
        print(f'  {CN[i]:>6} ' + ' '.join(f'{cm[i,j]:>6}' for j in range(5)), flush=True)

    # ---- 4. Generate test submission ----
    if has_test and test_probs_per_seed:
        test_probs_list = [test_probs_per_seed[s] for s in loaded_seeds]

        # Apply best strategy to test
        if best_name == 'arithmetic':
            final_test_probs = arithmetic_mean(test_probs_list)
        elif best_name == 'geometric':
            final_test_probs = geometric_mean(test_probs_list)
        else:
            final_test_probs = rank_average(test_probs_list)

        final_preds = final_test_probs.argmax(1)

        # Also save arithmetic mean probs (most common for downstream ensembling)
        arith_test_probs = arithmetic_mean(test_probs_list)

        # Submission
        sub = pd.DataFrame({'row_id': te_ids, 'class_id': final_preds})
        sub_path = f'T2_enormous_{n_loaded}seed_{best_name}_submission.csv'
        sub.to_csv(sub_path, index=False)

        # Save probs for future ensembling with Large models
        np.save('enormous_ensemble_test_probs.npy', arith_test_probs)
        np.save('enormous_ensemble_val_probs.npy', arith_probs)

        # Also save individual seed probs (useful for debugging)
        for s in loaded_seeds:
            np.save(f'enormous_seed{s}_test_probs.npy', test_probs_per_seed[s])

        # Distribution analysis
        dist = dict(zip(*np.unique(final_preds, return_counts=True)))
        max_conf = final_test_probs.max(axis=1)

        print(f'\n{"="*70}', flush=True)
        print(f'  SUBMISSION: {sub_path}', flush=True)
        print(f'  Strategy: {best_name} | Seeds: {loaded_seeds}', flush=True)
        print(f'  Predictions: {len(sub)} rows', flush=True)
        print(f'  Distribution: {" | ".join(f"{CN[k]}:{v}" for k,v in sorted(dist.items()))}',
              flush=True)
        print(f'  Confidence: mean={max_conf.mean():.3f} | '
              f'>0.95: {(max_conf>0.95).sum()} ({100*(max_conf>0.95).mean():.1f}%) | '
              f'<0.50: {(max_conf<0.50).sum()} ({100*(max_conf<0.50).mean():.1f}%)',
              flush=True)

        # Also generate arithmetic-mean submission if it wasn't the best
        if best_name != 'arithmetic':
            arith_preds = arith_test_probs.argmax(1)
            arith_sub = pd.DataFrame({'row_id': te_ids, 'class_id': arith_preds})
            arith_path = f'T2_enormous_{n_loaded}seed_arithmetic_submission.csv'
            arith_sub.to_csv(arith_path, index=False)
            print(f'  Also saved: {arith_path} (for LB comparison)', flush=True)

        print(f'  Saved: enormous_ensemble_test_probs.npy (for Large+Enormous ensemble)',
              flush=True)
    else:
        print('\n  No test data — skipping submission.', flush=True)

    total = (time.time() - T0) / 60
    print(f'\n{"="*70}', flush=True)
    print(f'  ✅ Done in {total:.1f} min', flush=True)
    print(f'{"="*70}', flush=True)
